_We'll build this together, live. This session's main exercises covered breadth-first search on unweighted graphs; this application applies BFS to real deal data._

## Application: Correlated Exposure in IPO Underwriting Syndicates

When several investment banks jointly underwrite an IPO, they don't just share the deal fees — they share **risk**. Each bank in the syndicate is contractually on the hook to buy its allocated block of shares, and all of them share reputational and legal exposure if the deal goes badly (a lawsuit over disclosures, a botched pricing, a post-IPO collapse). So a problem at one bank doesn't stay contained to that bank — it can spill over to everyone it has worked deals with.

`data/ipo_underwriter_syndicates.csv` in this folder is real data: the **actual underwriter syndicates** for 8 recent, real IPOs (Arm Holdings, Instacart, Birkenstock, Klaviyo, Cava Group, Astera Labs, Viking Holdings, Amer Sports), extracted directly from each company's SEC prospectus (Form 424B4) filing — 60 real investment banks, 133 real (bank, deal) pairs.

<img src="images/ipo_network.png" alt="Real IPO underwriter syndicate network" width="700">

**The question**: suppose one bank suffers a reputational or financial hit. Which *other* banks carry indirect exposure, even if they never worked a deal with the troubled bank directly? A bank two hops away shares a co-underwriter with them; a bank four hops away is separated by a longer chain of shared deals. This is exactly the kind of contagion question BFS answers — same mechanism as "degrees of separation," just tracing risk exposure instead of friendship.

### A note on tools

This case study runs breadth-first search on a real network, the same idea as your class-8 exercise. But instead of importing the `Graph`/`bfs` code you wrote there, we'll use **`networkx`**, Python's standard graph library.

Why not reuse your own code? This case study is meant to apply the *idea* of BFS to a new, real problem, not to test whether your class-8 implementation is correct, or depend on you having finished it yet. Using `networkx` also reflects how BFS actually gets used outside this course: `networkx` is the standard graph library in Python, so this is a chance to see the same algorithm through a professional tool instead of hand-rolled code.

Three things cover everything you need here:

- `networkx.Graph()`: creates an undirected graph, the same idea as the `Graph` class from class 8. Add edges with `.add_edge(u, v)`.
- `G.adj` (or equivalently `dict(G.adjacency())`): a dictionary mapping each node to its neighbors -- the same adjacency-list idea as the `{'a': {'b', 'c'}, ...}` dictionaries you built by hand in class 8.
- `networkx.single_source_shortest_path_length(G, start)`: runs BFS from `start` and returns a `{node: distance}` dictionary of everything reachable, exactly like the `bfs(graph, start)` function from class 8.

```python
import networkx as nx

G = nx.Graph()
G.add_edge('a', 'b')

print(G.adj['a'])  # {'b': {}}

distances = nx.single_source_shortest_path_length(G, 'a')
```

In [5]:
import networkx as nx

G = nx.Graph()
G.add_edge('a', 'b')

print(G.adj['a'])  # {'b': {}}

# distances = nx.single_source_shortest_path_length(G, 'a')
G.adj

{'b': {}}


AdjacencyView({'a': {'b': {}}, 'b': {'a': {}}})

## Part 1: Tracing exposure from one bank

**Step 1: build the graph.** Read `data/ipo_underwriter_syndicates.csv` and add an edge between each bank and each deal it was part of, using `networkx.Graph`.

**Hint:** `csv.DictReader` reads each row of a CSV file as a dictionary, keyed by the column names in the header row. Here the header is `bank,deal`, so each row becomes `{'bank': ..., 'deal': ...}`:

In [8]:
import csv

with open('data/ipo_underwriter_syndicates.csv') as f:
    reader = csv.DictReader(f)
    for row in reader:
        print(row)
        break
          # just show the first row

{'bank': 'Academy Securities, Inc', 'deal': 'Birkenstock'}


In [ ]:
import csv
import networkx as nx
import matplotlib

def build_syndicate_graph(filename):
    """Builds an undirected graph connecting each bank to the deals it underwrote."""
    g = nx.Graph()
    with open(filename) as f:
        reader = csv.DictReader(f)
        for row in reader:
            g.add_edge(row['bank'],row['deal'])
            
    return g

syndicate_graph = build_syndicate_graph('data/ipo_underwriter_syndicates.csv')

nx.draw(syndicate_graph, with_labels=True)

**Explore what you got back.** Whenever a package function returns something new, it's worth pausing to look at it directly -- its type, a sample of its contents -- before assuming its shape from memory. That's the normal way of working with any library, not just `networkx`.

In [13]:
print(type(syndicate_graph))
print(list(syndicate_graph.nodes)[:5])
print(list(syndicate_graph.edges)[:5])
print(syndicate_graph.adj['Needham & Company, LLC'])  # adjacency dict: neighbors of this one bank

<class 'networkx.classes.graph.Graph'>
['Academy Securities, Inc', 'Birkenstock', 'BMO Capital Markets Corp', 'Arm Holdings', 'BNP Paribas Securities Corp']
[('Academy Securities, Inc', 'Birkenstock'), ('Birkenstock', 'BMO Capital Markets Corp'), ('Birkenstock', 'BNP Paribas Securities Corp'), ('Birkenstock', 'BofA Securities, Inc'), ('Birkenstock', 'Citigroup Global Markets Inc')]
{'Astera Labs': {}, 'Klaviyo': {}}


**Step 2: run BFS.** Use `networkx.single_source_shortest_path_length(graph, start)` — it returns a dictionary of distances from `start` to every node it can reach, exactly like the `bfs` function from class 8. Let's use it to trace exposure outward from a single bank.

In [18]:
# your code here — compute exposure distances from 'Needham & Company, LLC'
import networkx
distances = networkx.single_source_shortest_path_length(syndicate_graph, 'Needham & Company, LLC')

**Same idea here:** look at what `distances` actually is before using it.

In [20]:
print(type(distances))
print(list(distances.items())[:5])
distances['Morgan Stanley & Co. LLC']

<class 'dict'>
[('Needham & Company, LLC', 0), ('Astera Labs', 1), ('Klaviyo', 1), ('Barclays Capital Inc', 2), ('Craig-Hallum Capital Group LLC', 2)]


2

**Step 3: read off an exposure distance.** Every path alternates bank → deal → bank → ..., so the distance to another *bank* is always an even number of hops. Divide by 2 to get "how many shared-deal links apart" two banks are — their degree of indirect exposure.

In [ ]:
target = 'Truist Securities, Inc'

print(distances[target]/2)
# your code here — print the number of hops from Needham & Company, LLC to target


<class 'str'>
<class 'dict'>
1.0


## Part 2: Your turn

Using `syndicate_graph` and `distances`, answer:

1. Which bank sits at the **outer edge** of Needham & Company, LLC's exposure network — still connected, but only through the longest chain of shared deals (largest BFS distance)? Since correlated risk weakens with each hop, this is the *weakest* exposure of all the banks still reachable.
2. Is the syndicate network **fully connected** — i.e. does BFS from `'Needham & Company, LLC'` reach all 60 banks? What would it mean for systemic risk if it weren't?

(Hint: every node in `syndicate_graph.nodes` is either a bank or a deal — you'll need to filter to just the bank names, e.g. by checking against the `bank` column of the CSV.)

In [ ]:
import csv
import networkx
import matplotlib



#Lets change the original function to output a set that includes only banks so we are looking for only BANKS  

def build_syndicate_graph(filename):
    """Builds an undirected graph connecting each bank to the deals it underwrote."""
    g = nx.Graph()
    banks =set()
    with open(filename) as f:
        reader = csv.DictReader(f)
        for row in reader:
            g.add_edge(row['bank'],row['deal'])
            banks.add(row['bank'])
            
    return banks, g

banks, syndicate_graph = build_syndicate_graph('data/ipo_underwriter_syndicates.csv')

distances = networkx.single_source_shortest_path_length(syndicate_graph, 'Needham & Company, LLC')
distances_bank = {}

for node in distances:
    if node in banks:
        distances_bank[node] = distances[node]/2

print(distances_bank)
if len(distances_bank) == len(banks):
    print("This Graph IS Connected")
    
    

{'Needham & Company, LLC': 0.0, 'Barclays Capital Inc': 1.0, 'Craig-Hallum Capital Group LLC': 1.0, 'Deutsche Bank Securities Inc': 1.0, 'Evercore Group L.L.C': 1.0, 'J.P. Morgan Securities LLC': 1.0, 'Jefferies LLC': 1.0, 'Loop Capital Markets LLC': 1.0, 'Morgan Stanley & Co. LLC': 1.0, 'Roth Capital Partners, LLC': 1.0, 'Siebert Williams Shank & Co., LLC': 1.0, 'Stifel, Nicolaus & Company, Incorporated': 1.0, 'Canaccord Genuity LLC': 1.0, 'Citigroup Global Markets Inc': 1.0, 'Cowen and Company, LLC': 1.0, 'Mizuho Securities USA LLC': 1.0, 'Piper Sandler & Co': 1.0, 'Robert W. Baird & Co. Incorporated': 1.0, 'Truist Securities, Inc': 1.0, 'William Blair & Company, L.L.C': 1.0, 'Blaylock Van, LLC': 2.0, 'BofA Securities, Inc': 2.0, 'Citizens JMP Securities, LLC': 2.0, 'Drexel Hamilton, LLC': 2.0, 'LionTree Advisors LLC': 2.0, 'Oppenheimer & Co. Inc': 2.0, 'R. Seelaus & Co., LLC': 2.0, 'Samuel A. Ramirez & Company, Inc': 2.0, 'SoFi Securities LLC': 2.0, 'Stern Brothers & Co': 2.0, 'Tigr

True

## Part 3: Extension: how small is this world?

So far we've only measured distances *from* `'Needham & Company, LLC'`. To find the distance between two banks where **neither** is your BFS starting point, you need to rerun BFS from one of them — a single BFS run only gives correct distances for pairs that include its own starting node.

To get the shortest distance between **every** pair of banks, run BFS once from *each* bank and keep every result. Then build a **histogram**: count how many bank pairs fall at each distance. The largest distance with a nonzero count is the network's **diameter** — the worst-case degree of separation anywhere in this syndicate ecosystem.

In [ ]:
def all_pairs_distances(graph, nodes):
    """Computes shortest-path distances between every pair of nodes in `nodes`."""
    all_pairs = {}
    # your code here

    return all_pairs

bank_pairs = all_pairs_distances(syndicate_graph, banks)

# your code here — build a dictionary counting how many bank pairs are at each distance
# (excluding self-pairs), then print the histogram and the maximum (the network's diameter)